# EDA — CarDD Dataset

**Vereist**: run eerst `python src/prepare_data.py` zodat de data in
`data/processed/cardd_yolo/` staat (YOLO-formaat, geëxporteerd via FiftyOne).

Doel van dit notebook:
- Class balance bekijken (6 schadecategorieën)
- Resolutie/aspect ratio variatie
- Voorbeeldafbeeldingen per klasse visualiseren (met masks)
- Corrupte/verkeerd gelabelde afbeeldingen opsporen

In [ ]:
import yaml
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

DATA_DIR = Path('../data/processed/cardd_yolo')

with open(DATA_DIR / 'dataset.yaml') as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg['names']
print('Klassen:', class_names)

## 1. Class balance

In YOLO-segmentatieformaat begint elke regel in een `.txt`-labelbestand met
de class_id, gevolgd door de polygoon-coördinaten van het mask.

In [ ]:
counter = Counter()
for split in ['train', 'val']:
    labels_dir = DATA_DIR / split / 'labels'
    if not labels_dir.exists():
        continue
    for label_file in labels_dir.glob('*.txt'):
        with open(label_file) as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    counter[class_id] += 1

df_counts = pd.DataFrame(
    [(class_names[k], v) for k, v in counter.items()],
    columns=['class', 'count']
).sort_values('count', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=df_counts, x='count', y='class', palette='viridis')
plt.title('Class balance — CarDD (train + val)')
plt.tight_layout()
plt.show()
df_counts

**Observatie om in je verslag/presentatie te noemen**: als bepaalde klassen
(bijv. `tire flat`, `glass shatter`) veel minder voorkomen dan `scratch`/`dent`,
verklaart dat waarom het model daar slechter op scoort (lagere recall).

## 2. Resolutie & aspect ratio

In [ ]:
sizes = []
corrupt_files = []
images_dir = DATA_DIR / 'train' / 'images'

for img_path in list(images_dir.glob('*'))[:500]:  # sample voor snelheid
    try:
        with Image.open(img_path) as img:
            sizes.append(img.size)
    except Exception as e:
        corrupt_files.append((img_path, str(e)))

df_sizes = pd.DataFrame(sizes, columns=['width', 'height'])
print(f'Corrupte bestanden gevonden: {len(corrupt_files)}')
df_sizes.describe()

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df_sizes['width'], df_sizes['height'], alpha=0.4, s=10)
plt.xlabel('Breedte (px)')
plt.ylabel('Hoogte (px)')
plt.title('Resolutieverdeling (sample van 500 afbeeldingen)')
plt.show()

## 3. Voorbeeldafbeeldingen per klasse

Handig om te controleren of de labels visueel kloppen — dit is precies
wat een schade-expert zou moeten valideren voordat je gaat trainen.

In [ ]:
import random

labels_dir = DATA_DIR / 'train' / 'labels'
label_files = list(labels_dir.glob('*.txt'))
sample_files = random.sample(label_files, min(6, len(label_files)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, label_file in zip(axes.flat, sample_files):
    img_file = images_dir / (label_file.stem + '.jpg')
    if not img_file.exists():
        img_file = images_dir / (label_file.stem + '.png')
    if img_file.exists():
        img = Image.open(img_file)
        ax.imshow(img)
        with open(label_file) as f:
            classes_in_img = {class_names[int(l.split()[0])] for l in f if l.strip()}
        ax.set_title(', '.join(classes_in_img), fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Conclusie van de EDA

TODO na het runnen: noteer hier kort
- Welke klasse(n) ondervertegenwoordigd zijn
- Of er corrupte bestanden zijn gevonden
- Of de resolutie consistent genoeg is voor training
- Of de visuele steekproef klopt met de labels